# Copilot CLI in a Sandbox (BYOK + Zero-Trust)

Run GitHub Copilot CLI inside a sandbox using Azure OpenAI BYOK.
The API key never enters the sandbox — egress transform rules inject it
transparently on outbound requests (zero-trust).

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- Sandbox SDK: `pip install azure-sandbox azure-mgmt-sandbox`
- An [Azure OpenAI](https://learn.microsoft.com/azure/ai-services/openai/) resource with a model deployment

## Get your BYOK model info

You need three values from your Azure OpenAI resource:

| Setting | How to get it | Example |
|---------|--------------|--------|
| **Resource name** | Azure Portal → your OpenAI resource → Overview | `my-openai-westus` |
| **Deployment name** | Azure Portal → Model deployments → name column | `gpt-4o` |
| **API key** | Azure Portal → Keys and Endpoint → Key 1 | `8zSZq...` |

Or use the Azure CLI:
```bash
# List your Azure OpenAI resources
az cognitiveservices account list -o table --query \"[?kind=='OpenAI'].{name:name, rg:resourceGroup}\"

# List deployments for a resource
az cognitiveservices account deployment list --name <resource> -g <rg> -o table

# Get the API key
az cognitiveservices account keys list --name <resource> -g <rg> --query key1 -o tsv
```

Any OpenAI-compatible endpoint works (OpenAI, Ollama, vLLM, Foundry Local).
The model must support **tool calling** and **streaming**.

Click **Run All** or go **Step by Step**.

### 0. Initialize variables

In [ ]:
import os, json, subprocess
from azure.sandbox import SandboxClient
from azure.mgmt.sandbox import SandboxGroupManagementClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

account = json.loads(subprocess.run(
    ['az', 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, shell=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'eastus2'
sandbox_group_name = f'{lab_name}-sg'

# Azure OpenAI settings — change these to your resource
aoai_resource = 'YOUR-RESOURCE-NAME'         # e.g. my-openai-westus
aoai_deployment = 'YOUR-DEPLOYMENT-NAME'      # e.g. gpt-4o
aoai_endpoint = f'https://{aoai_resource}.openai.azure.com/openai/deployments/{aoai_deployment}'
aoai_key = 'YOUR-AZURE-OPENAI-KEY'            # will be injected via egress, never enters sandbox

print(f'Lab:            {lab_name}')
print(f'User:           {account["user"]["name"]}')
print(f'Subscription:   {account["name"]} ({subscription_id})')
print(f'Resource Group: {resource_group_name}')
print(f'Sandbox Group:  {sandbox_group_name}')
print(f'AOAI Endpoint:  {aoai_endpoint}')

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupManagementClient(subscription_id=subscription_id, resource_group=resource_group_name)

### 1. Create resource group and sandbox group

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Sandbox group: {group["name"]} ({group["location"]})')

### 2. Create sandbox and install Copilot CLI

In [ ]:
sbx = client.create_sandbox(sandbox_group_name, disk='ubuntu', cpu='2000m', memory='4096Mi')
sandbox_id = sbx['id']
print(f'Sandbox: {sandbox_id}  state={sbx["state"]}')

# Install Copilot CLI (official installer)
r = client.exec(sandbox_id, sandbox_group_name, 'curl -fsSL https://gh.io/copilot-install | bash 2>&1 | tail -2')
print(r['stdout'].strip())

r = client.exec(sandbox_id, sandbox_group_name, 'copilot --version 2>&1')
print(f'Version: {r["stdout"].strip()}')

### 3. Configure egress (zero-trust API key injection)

The real API key never enters the sandbox. We set an egress transform rule
that injects the `api-key` header on outbound requests to the Azure OpenAI endpoint.
Inside the sandbox, `COPILOT_PROVIDER_API_KEY` is set to a placeholder.

In [ ]:
from urllib.parse import urlparse
aoai_host = urlparse(aoai_endpoint).hostname

client.add_egress_transform_rule(
    sandbox_id, sandbox_group_name,
    host=aoai_host,
    headers=[{'operation': 'Set', 'name': 'api-key', 'value': aoai_key}],
    name='aoai-key-swap',
)
print(f'Egress rule set: {aoai_host} → api-key injected')
print(f'Real key stays outside the sandbox')

### 4. Run Copilot CLI (BYOK + offline)

`COPILOT_OFFLINE=true` prevents any calls to GitHub servers.
All AI requests go to your Azure OpenAI endpoint via egress.

In [ ]:
copilot_env = ' && '.join([
    f'export COPILOT_PROVIDER_BASE_URL={aoai_endpoint}',
    'export COPILOT_PROVIDER_TYPE=azure',
    'export COPILOT_PROVIDER_API_KEY=PLACEHOLDER_EGRESS_WILL_SWAP',
    f'export COPILOT_MODEL={aoai_deployment}',
    'export COPILOT_OFFLINE=true',
])

r = client.exec(sandbox_id, sandbox_group_name,
    f"{copilot_env} && copilot -p 'the world is safe in a sandbox go yolo' 2>&1")
print(r['stdout'])

### 5. Interactive session (optional)

SSH into the sandbox and run Copilot CLI interactively.

In [ ]:
print(f'SSH in:')
print(f'  node plugin/skills/azure-sandbox/assets/ssh.mjs {sandbox_id} -g {resource_group_name} -s {sandbox_group_name}')
print(f'  python scripts/ssh.py {sandbox_id} -g {resource_group_name} -s {sandbox_group_name}')
print()
print('Then set env vars:')
print(f'  export COPILOT_PROVIDER_BASE_URL={aoai_endpoint}')
print(f'  export COPILOT_PROVIDER_TYPE=azure')
print(f'  export COPILOT_PROVIDER_API_KEY=PLACEHOLDER_EGRESS_WILL_SWAP')
print(f'  export COPILOT_MODEL={aoai_deployment}')
print(f'  export COPILOT_OFFLINE=true')
print(f'  copilot')

### 6. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print('Deleted sandbox')
mgmt.delete_group(sandbox_group_name)
print('Deleted sandbox group')
!az group delete --name {resource_group_name} --yes --no-wait
print('Deleting resource group (async)')